# Parâmetros do notebook

In [0]:
dbutils.widgets.text("environment", "dev", "Environment")

environment = dbutils.widgets.get("environment")

print("Environment:", environment)

In [0]:
dbutils.widgets.text("id_projeto", "endometriose", "ID Projeto")
id_projeto = dbutils.widgets.get("id_projeto")
print("id_projeto:", id_projeto)

In [0]:
if environment in ["hml", "prd"]:
    environment_tbl = ""
else:
    environment_tbl = f"{environment}_"

print(f"environment_tbl: {environment_tbl}")

In [0]:
root_path_gold = f"/mnt/trusted/datalake/ia/data/{id_projeto}/gold/"
print(f"root_path_gold: {root_path_gold}")

# DDL das tabelas e views

## Variáveis com os nomes das tabelas

In [0]:
table_name_entrada = f"{environment_tbl}tbl_gold_modelo_{id_projeto}_entrada"
table_name_pre_processamento = f"{environment_tbl}tbl_gold_modelo_{id_projeto}_pre_processamento"
table_name_saida = f"{environment_tbl}tbl_gold_modelo_{id_projeto}_saida"
view_name = f"{environment_tbl}vw_gold_modelo_{id_projeto}"
table_name_retorno = f"{environment_tbl}tbl_gold_modelo_{id_projeto}_retorno"
table_name_retorno_hist = f"{table_name_retorno}_historico"
view_name_analise_retorno = f"{environment_tbl}vw_gold_modelo_{id_projeto}_analise_retorno"
table_name_navegacao = f"{environment_tbl}tbl_gold_modelo_{id_projeto}_navegacao"

In [0]:
print(f"{'table_name_entrada':40}: {table_name_entrada}")
print(f"{'table_name_pre_processamento':40}: {table_name_pre_processamento}")
print(f"{'table_name_saida':40}: {table_name_saida}")
print(f"{'view_name':40}: {view_name}")
print(f"{'table_name_retorno':40}: {table_name_retorno}")
print(f"{'table_name_retorno_hist':40}: {table_name_retorno_hist}")
print(f"{'view_name_analise_retorno':40}: {view_name_analise_retorno}")
print(f"{'table_name_navegacao':40}: {table_name_navegacao}")


## tbl_gold_entrada

In [0]:
table_name_entrada

In [0]:
# spark.sql(f"drop table if exists ia.{table_name_entrada}").display()
# dbutils.fs.rm(f"{root_path_gold}{table_name_entrada}", recurse=True)

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS ia.{table_name_entrada} (
        dataExecucaoModelo DATE,
        idPredicao STRING,
        idExame STRING,
        dataExame TIMESTAMP,
        dataLiberacaoLaudo TIMESTAMP,
        nomeConvenio STRING,
        tipoCodigo STRING,
        codigo STRING,
        nomeCodigo STRING,
        descricaoProcedimento STRING,
        idUnidade STRING,
        nomeUnidade STRING,
        cnpjUnidade STRING,
        regionalUnidade STRING,
        tipoUnidade STRING,
        idMedico STRING,
        numCrm STRING,
        ufCrm STRING,
        nomeMedico STRING,
        idPaciente STRING,
        cpfPaciente STRING,
        nomePaciente STRING,
        dataNascimentoPaciente TIMESTAMP,
        idadePaciente BIGINT,
        sexoPaciente STRING,
        telefoneContato STRING,
        laudoOriginal STRING,
        laudoDuplicado INTEGER,
        dataCarga TIMESTAMP
    )
    PARTITIONED BY (dataExecucaoModelo)
    LOCATION '{root_path_gold}{table_name_entrada}'
""").display()

## tbl_gold_pre_processamento

In [0]:
table_name_pre_processamento

In [0]:
# spark.sql(f"drop table if exists ia.{table_name_pre_processamento}").display()
# dbutils.fs.rm(f"{root_path_gold}{table_name_pre_processamento}", recurse=True)

In [0]:
spark.sql(
    f"""
    create table if not exists ia.{table_name_pre_processamento}
    (
        dataExecucaoModelo DATE,
        idPredicao STRING,
        idExame STRING,
        idPaciente STRING,
        descricaoProcedimento STRING,
        laudoOriginal STRING,
        dataExame DATE,
        laudoOriginalLimpo STRING,
        laudoTratado STRING,
        listaSintomas ARRAY<STRING>,
        listaAchados ARRAY<STRING>,
        classificacao STRING,
        laudoResumido STRING
    )
    partitioned by (dataExecucaoModelo)
    location '{root_path_gold}{table_name_pre_processamento}'
"""
).display()

## tbl_gold_saida

In [0]:
table_name_saida

In [0]:
# spark.sql(f"drop table if exists ia.{table_name_saida}").display()
# dbutils.fs.rm(f"{root_path_gold}{table_name_saida}", recurse=True)

In [0]:
spark.sql(f"""
    create table if not exists ia.{table_name_saida}
    (
        dataExecucaoModelo DATE,
        idPredicao STRING,
        idExame STRING,
        idPaciente STRING,
        descricaoProcedimento STRING,
        laudoOriginal STRING,
        laudoResumido STRING,
        listaSintomas ARRAY<STRING>,
        listaAchados ARRAY<STRING>,
        classificacao STRING,
        dataExame DATE,
        classificacaoEndometriose STRING
    )
    
    partitioned by (dataExecucaoModelo)
    location '{root_path_gold}{table_name_saida}'
""").display()

## vw_gold_entrada_saida

In [0]:
view_name

In [0]:
# spark.sql(f"drop view if exists ia.{view_name}").display()

In [0]:
spark.sql(f"""
    create or replace view ia.{view_name}
    as
    select
        ent.dataExecucaoModelo,
        ent.idPredicao,
        ent.idExame,
        ent.dataExame,
        ent.dataLiberacaoLaudo,
        ent.nomeConvenio,
        ent.tipoCodigo,
        ent.codigo,
        ent.nomeCodigo,
        ent.descricaoProcedimento,
        ent.idUnidade,
        ent.nomeUnidade,
        ent.tipoUnidade,
        ent.idMedico,
        ent.ufCrm,
        default.unencrypt(ent.cnpjUnidade) as cnpjUnidade,
        default.unencrypt(ent.regionalUnidade) as regionalUnidade,
        default.unencrypt(ent.numCrm) as numCrm,
        if(trim(ent.nomeMedico) != "", default.unencrypt(ent.nomeMedico), null) as nomeMedico,
        ent.idPaciente,
        default.unencrypt(ent.cpfPaciente) as cpfPaciente,
        default.unencrypt(ent.nomePaciente) as nomePaciente,
        ent.dataNascimentoPaciente,
        ent.idadePaciente,
        ent.sexoPaciente,
        ent.telefoneContato,
        ent.laudoOriginal,
        ent.laudoDuplicado,
        ent.dataCarga,

        pre.laudoOriginalLimpo,
        pre.laudoTratado,
        pre.listaSintomas,
        pre.listaAchados,
        pre.classificacao,
        pre.laudoResumido,
    
        sai.classificacaoEndometriose

    from ia.{table_name_entrada} as ent

    left join ia.{table_name_pre_processamento} as pre
        on ent.idPredicao = pre.idPredicao

    left join ia.{table_name_saida} as sai
        on ent.idPredicao = sai.idPredicao
""").display()

## tbl_gold_retorno

In [0]:
table_name_retorno

In [0]:
# spark.sql(f"drop table if exists ia.{table_name_retorno}").display()
# dbutils.fs.rm(f"{root_path_gold}{table_name_retorno}", recurse=True)

# # Criada dinamicamente, qdo o notebook de retorno for processado
# spark.sql(f"drop table if exists ia.{table_name_retorno_hist}").display()
# dbutils.fs.rm(f"{root_path_gold}{table_name_retorno_hist}", recurse=True)

In [0]:
spark.sql(f"""
    create table if not exists ia.{table_name_retorno}
    (
        idRetorno string,
        dataExecucaoModelo date,
        idPredicao string,
        idExame string,
        dataHoraRetorno timestamp,
        dataHoraAtualizacao timestamp,
        arquivoRetorno string,
        achado string,
        observacoes string,
        destino string
    )
    partitioned by (dataExecucaoModelo)
    location '{root_path_gold}{table_name_retorno}'
""").display()

## vw_gold_analise_retorno

In [0]:
view_name_analise_retorno

In [0]:
# spark.sql(f"drop view if exists ia.{view_name_analise_retorno}").display()
# dbutils.fs.rm(f"{root_path_gold}{view_name_analise_retorno}", recurse=True)

In [0]:
table_name_pre_processamento

In [0]:
%sql
desc ia.dev_tbl_gold_modelo_endometriose_saida

In [0]:
spark.sql(f"""
    create or replace view ia.{view_name_analise_retorno}
    as
    select
        ent.dataExecucaoModelo,
        ent.idPredicao,
        ent.idExame,
        ent.dataExame,
        ent.dataLiberacaoLaudo,
        ent.nomeConvenio,
        ent.tipoCodigo,
        ent.codigo,
        ent.nomeCodigo,
        ent.descricaoProcedimento,
        ent.idUnidade,
        ent.nomeUnidade,
        default.unencrypt(ent.cnpjUnidade, 0) as cnpjUnidade,
        default.unencrypt(ent.regionalUnidade, 0) as regionalUnidade,
        ent.tipoUnidade,
        ent.idMedico,
        default.unencrypt(ent.numCrm, 0) as numCrm,
        ent.ufCrm,
        default.unencrypt(ent.nomeMedico, 0) as nomeMedico,
        ent.idPaciente,
        default.unencrypt(ent.cpfPaciente, 0) as cpfPaciente,
        default.unencrypt(ent.nomePaciente, 0) as nomePaciente,
        ent.dataNascimentoPaciente,
        ent.idadePaciente,
        ent.sexoPaciente,
        ent.telefoneContato,
        ent.laudoOriginal,
        ent.laudoDuplicado,
        ent.dataCarga,
        
        pre.laudoOriginalLimpo,
        pre.laudoTratado,
        pre.listaSintomas,
        pre.listaAchados,
        pre.classificacao,
        pre.laudoResumido,
        
        sai.classificacaoEndometriose,

        ret.idRetorno,
        -- ret.dataExecucaoModelo,
        -- ret.idPredicao,
        -- ret.idExame,
        ret.dataHoraRetorno,
        ret.dataHoraAtualizacao,
        ret.arquivoRetorno,
        ret.achado,
        ret.observacoes,
        ret.destino

    from ia.{table_name_entrada} as ent

    left join ia.{table_name_pre_processamento} as pre
        on ent.idPredicao = pre.idPredicao

    left join ia.{table_name_saida} as sai
        on ent.idPredicao = sai.idPredicao

    left join ia.{table_name_retorno} as ret
        on ent.idPredicao = ret.idPredicao

""").display()

## tbl_gold_navegacao

In [0]:
table_name_navegacao

In [0]:
# spark.sql(f"drop table if exists ia.{table_name_navegacao}").display()
# dbutils.fs.rm(f"{root_path_gold}{table_name_navegacao}", recurse=True)

In [0]:
spark.sql(f"""
    create table if not exists ia.{table_name_navegacao}
    (
        idNavegacao string,
        dataEnviadoNavegacao date,
        dataExecucaoModelo date,
        idPredicao string,
        idExame string
    )
    partitioned by (dataEnviadoNavegacao)
    location '{root_path_gold}{table_name_navegacao}'
""").display()